# Relational Graph Transformer BASELINE

A clean, minimal RGT baseline on the **time-respecting Elliptic++ graph**
with **raw node features only**. No trajectory features, no PPR features --
just the graph and the original Elliptic++ tx + your causally-recomputed
wallet features. Same architecture, same per-target causality contract as
`rgt.ipynb` -- the only difference is the input feature stack.

## What this measures

How much can the relational graph transformer learn from JUST:
- The full causal heterogeneous graph (bipartite + wallet-wallet + tx-tx)
- Raw 182-dim tx features
- 55-dim causal cumulative wallet features

versus what the *engineered-feature* RGT (`rgt.ipynb`) achieves with
+103 trajectory + 5 PPR features added on top of tx nodes. The delta tells
us how much of the trajectory/PPR signal the GNN can recover from the
graph structure alone.

## Causality contract -- strict per-target everywhere

For target tx τ at time `t = k`, the model is forward-passed on the
**per-target snapshot** `g_k = build_snapshot(k)`:
- All edges have `edge_t ≤ k`
- Wallet features are causal cumulative through `k` (from
  `wallets_features_causal.csv`)
- Tx features are static (raw 182, time-independent)

Snapshot construction itself is the time-respecting attention mask -- HGTConv
attends only over the sampled subgraph drawn from `g_k`, so by construction
τ-at-k cannot attend to future-of-k information.

## Tx node features (182 -- raw only)

From `txs_features.csv`. Per-tx, time-independent. **No engineered features.**

## Wallet node features (55)

Causal cumulative wallet features through `k` from `wallets_features_causal.csv`.

## Edges (full causal graph)

- **bipartite** `wallet ↔ tx`: `host_t ≤ k`
- **wallet ↔ wallet** (AddrAddr causal): `first_t(e) ≤ k`, undirected
- **tx ↔ tx** (within-step): `t_edge ≤ k`, undirected

## Architecture (same as `rgt.ipynb`)

`HIDDEN=96`, `HEADS=2`, `N_LAYERS=2`, `DROPOUT=0.4`, `LR=3e-4`,
gradient clipping, Pre-LN, AdamW, NeighborLoader sampling, bf16 autocast
with the HGTConv-around-fp32 workaround.

In [ ]:
# Cell 1: imports, paths, config

import os, sys, time, json, math, copy, gc
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, precision_recall_curve
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv
from torch_geometric.loader import NeighborLoader

# --- Path resolution: works on Colab (drive mount) and local ---
COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    COLAB = True
    REPO = "/content/drive/MyDrive/stat175-final-project"
    ACTORS = os.path.join(REPO, "actor_data")
    TX = os.path.join(REPO, "data")
except Exception:
    REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
    if not os.path.exists(os.path.join(REPO, "actors_data")):
        REPO = "/Users/jarayliu/Documents/GitHub/stat175-final"
    ACTORS = os.path.join(REPO, "actors_data")
    TX = os.path.join(REPO, "transactions_data")
print("repo:", REPO, " colab:", COLAB)

# --- Reproducibility ---
RANDOM_SEED = 175
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# --- Time-window protocol ---
N_TIMESTEPS = 49
TRAIN_END = 29
VAL_END = 34
TEST_START = 35
TEST_TIMESTEPS = list(range(TEST_START, N_TIMESTEPS + 1))
WALK_VAL_WIN = 4

# --- Model + training config (same as rgt.ipynb) ---
HIDDEN = 96
N_LAYERS = 2
HEADS = 2
DROPOUT = 0.4
ATTN_DROPOUT = 0.4
LR = 3e-4
WEIGHT_DECAY = 5e-4
GRAD_CLIP = 1.0
N_EPOCHS_STRICT = 50
N_EPOCHS_WALK = 15
PATIENCE = 12

# --- NeighborLoader sampling ---
SAMPLE_FANOUT = [10, 5]
SAMPLE_BATCH  = 2048

# --- bf16 mixed precision ---
USE_BF16 = True

# --- Device ---
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device:", DEVICE, " bf16:", USE_BF16 and DEVICE.type == "cuda")


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def amp_ctx():
    if USE_BF16 and DEVICE.type == "cuda":
        return torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


In [ ]:
# Cell 2: load all data

t0 = time.time()

tx_feat_df = pd.read_csv(os.path.join(TX, "txs_features.csv"))
tx_id_arr = tx_feat_df["txId"].values
tx_t_arr  = tx_feat_df["Time step"].values.astype(np.int64)
TX_FEAT_COLS = [c for c in tx_feat_df.columns if c not in ("txId", "Time step")]
tx_X_raw = tx_feat_df[TX_FEAT_COLS].fillna(0.0).values.astype(np.float32)
N_TX = len(tx_id_arr)
F_TX = tx_X_raw.shape[1]
print(f"tx: N={N_TX:,}  F_TX={F_TX}  t-range=[{tx_t_arr.min()}, {tx_t_arr.max()}]")

tx_cls_df = pd.read_csv(os.path.join(TX, "txs_classes.csv"))
cls_map = dict(zip(tx_cls_df["txId"].values, tx_cls_df["class"].values.astype(np.int64)))
tx_label = np.full(N_TX, -1, dtype=np.int64)
for i, txid in enumerate(tx_id_arr):
    c = cls_map.get(txid)
    if c == 1:   tx_label[i] = 1
    elif c == 2: tx_label[i] = 0
print(f"tx labels: illicit={(tx_label==1).sum():,}  licit={(tx_label==0).sum():,}  unknown={(tx_label==-1).sum():,}")

# --- Causal wallet features ---
def _find_csv(candidates):
    for p in candidates:
        if os.path.exists(p): return p
    return None

wf_path = _find_csv([
    os.path.join(ACTORS, "wallets_features_causal.csv"),
    os.path.join(ACTORS, "wallets_features_causal.txt"),
])
assert wf_path is not None, f"could not find wallets_features_causal in {ACTORS}"
wf = pd.read_csv(wf_path)
WF_FEAT_COLS = [c for c in wf.columns if c not in ("address", "Time step")]
F_W = len(WF_FEAT_COLS)
print(f"wallet causal feats: {len(wf):,} rows, F_W={F_W}")

# --- Bipartite ---
ai = pd.read_csv(os.path.join(ACTORS, "AddrTx_edgelist.txt"))
ao = pd.read_csv(os.path.join(ACTORS, "TxAddr_edgelist.txt"))
print(f"bipartite: in={len(ai):,}  out={len(ao):,}")

# --- AddrAddr causal ---
aa_path = _find_csv([
    os.path.join(ACTORS, "AddrAddr_edgelist_causal.csv"),
    os.path.join(ACTORS, "AddrAddr_edgelist_causal.txt"),
])
assert aa_path is not None, f"could not find AddrAddr_edgelist_causal in {ACTORS}"
aa = pd.read_csv(aa_path)
print(f"wallet-wallet edges: {len(aa):,}  first_t range=[{int(aa['first_t'].min())}, {int(aa['first_t'].max())}]")

# --- tx-tx ---
tt = pd.read_csv(os.path.join(TX, "txs_edgelist.csv"))
print(f"tx-tx edges: {len(tt):,}")

print(f"[load] {time.time()-t0:.1f}s")


In [ ]:
# Cell 3: node ID maps + per-edge timestamp arrays

t0 = time.time()

tx_to_idx = {txid: i for i, txid in enumerate(tx_id_arr)}
wallet_addrs = wf["address"].drop_duplicates().values
N_W = len(wallet_addrs)
w_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
print(f"N_TX={N_TX:,}  N_W={N_W:,}")

ai_w = np.array([w_to_idx[w] for w in ai["input_address"].values], dtype=np.int64)
ai_x = np.array([tx_to_idx[t] for t in ai["txId"].values], dtype=np.int64)
ai_t = tx_t_arr[ai_x]

ao_x = np.array([tx_to_idx[t] for t in ao["txId"].values], dtype=np.int64)
ao_w = np.array([w_to_idx[w] for w in ao["output_address"].values], dtype=np.int64)
ao_t = tx_t_arr[ao_x]

aa_in  = np.array([w_to_idx[w] for w in aa["input_address"].values],  dtype=np.int64)
aa_out = np.array([w_to_idx[w] for w in aa["output_address"].values], dtype=np.int64)
aa_t   = aa["first_t"].values.astype(np.int64)

tt_a = np.array([tx_to_idx[t] for t in tt["txId1"].values], dtype=np.int64)
tt_b = np.array([tx_to_idx[t] for t in tt["txId2"].values], dtype=np.int64)
tt_t = tx_t_arr[tt_a]
assert (tx_t_arr[tt_a] == tx_t_arr[tt_b]).all(), "tx-tx edges should be within-timestep"

print(f"  bipartite: in={len(ai_w):,}  out={len(ao_x):,}  walletXwallet={len(aa_in):,}  txXtx={len(tt_a):,}")
print(f"[node/edge build] {time.time()-t0:.1f}s")


In [ ]:
# Cell 4: wallet feature lookup table

t0 = time.time()

wf_sorted = wf.sort_values(["address", "Time step"]).reset_index(drop=True)
addr_arr = wf_sorted["address"].values
t_arr_wf = wf_sorted["Time step"].values.astype(np.int64)
WF_RAW = wf_sorted[WF_FEAT_COLS].fillna(0.0).values.astype(np.float32)

last_row_at_T = np.full((N_TIMESTEPS + 1, N_W), -1, dtype=np.int32)
addr_to_id = np.array([w_to_idx[a] for a in wf_sorted["address"].values], dtype=np.int64)
for j in range(len(wf_sorted)):
    last_row_at_T[t_arr_wf[j], addr_to_id[j]] = j
for T in range(1, N_TIMESTEPS + 1):
    mask = (last_row_at_T[T] == -1)
    last_row_at_T[T, mask] = last_row_at_T[T - 1, mask]
print(f"  last_row_at_T: shape={last_row_at_T.shape} ({last_row_at_T.nbytes/1024/1024:.0f} MB)")


def wallet_features_at_T(T_query: int) -> np.ndarray:
    idx = last_row_at_T[T_query]
    out = np.zeros((N_W, F_W), dtype=np.float32)
    mask = idx >= 0
    out[mask] = WF_RAW[idx[mask]]
    return out

print(f"[wallet feature lookup] {time.time()-t0:.1f}s")


In [ ]:
# Cell 5: standardize features (fit on TRAIN_END window only)
# Baseline uses RAW tx features only -- no traj/PPR concat.

t0 = time.time()

tx_train_mask = (tx_t_arr <= TRAIN_END)
tx_scaler = StandardScaler().fit(tx_X_raw[tx_train_mask])
TX_X = tx_scaler.transform(tx_X_raw).astype(np.float32)
print(f"tx_X std (raw {F_TX}): shape={TX_X.shape}  fit on {tx_train_mask.sum():,} train txs (t<={TRAIN_END})")

train_wf_mask = (t_arr_wf <= TRAIN_END)
wf_scaler = StandardScaler().fit(WF_RAW[train_wf_mask])
WF_RAW = wf_scaler.transform(WF_RAW).astype(np.float32)
print(f"wf std: fit on {train_wf_mask.sum():,} causal rows with t<={TRAIN_END}")

print(f"[standardize] {time.time()-t0:.1f}s")


In [ ]:
# Cell 6: heterogeneous graph snapshot builder
#
# build_snapshot(k) returns a HeteroData where every edge has edge_t <= k and
# wallet features are causal cumulative through k.
# Tx features are RAW 182 (no engineered features).

T_AI_W = torch.from_numpy(ai_w);  T_AI_X = torch.from_numpy(ai_x);  T_AI_T = torch.from_numpy(ai_t)
T_AO_X = torch.from_numpy(ao_x);  T_AO_W = torch.from_numpy(ao_w);  T_AO_T = torch.from_numpy(ao_t)
T_AA_I = torch.from_numpy(aa_in); T_AA_O = torch.from_numpy(aa_out); T_AA_T = torch.from_numpy(aa_t)
T_TT_A = torch.from_numpy(tt_a);  T_TT_B = torch.from_numpy(tt_b);  T_TT_T = torch.from_numpy(tt_t)
T_TX_X = torch.from_numpy(TX_X)


def build_snapshot(k: int, device=None) -> HeteroData:
    data = HeteroData()
    data["tx"].x = T_TX_X if device is None else T_TX_X.to(device, non_blocking=True)
    wf_T = torch.from_numpy(wallet_features_at_T(k))
    data["wallet"].x = wf_T if device is None else wf_T.to(device, non_blocking=True)

    m_in  = (T_AI_T <= k); m_out = (T_AO_T <= k)
    e_w_to_tx     = torch.stack([T_AI_W[m_in],  T_AI_X[m_in]],  dim=0)
    e_tx_to_w_rev = torch.stack([T_AI_X[m_in],  T_AI_W[m_in]],  dim=0)
    e_tx_to_w     = torch.stack([T_AO_X[m_out], T_AO_W[m_out]], dim=0)
    e_w_to_tx_rev = torch.stack([T_AO_W[m_out], T_AO_X[m_out]], dim=0)
    if device is not None:
        e_w_to_tx     = e_w_to_tx.to(device, non_blocking=True)
        e_tx_to_w_rev = e_tx_to_w_rev.to(device, non_blocking=True)
        e_tx_to_w     = e_tx_to_w.to(device, non_blocking=True)
        e_w_to_tx_rev = e_w_to_tx_rev.to(device, non_blocking=True)
    data["wallet", "sends_to", "tx"].edge_index    = e_w_to_tx
    data["tx",     "rev_sends_to", "wallet"].edge_index = e_tx_to_w_rev
    data["tx",     "pays_to", "wallet"].edge_index = e_tx_to_w
    data["wallet", "rev_pays_to", "tx"].edge_index = e_w_to_tx_rev

    # Wallet<->wallet co-tx: undirected (forward + reverse stacked, single relation)
    m_aa = (T_AA_T <= k)
    aa_src = torch.cat([T_AA_I[m_aa], T_AA_O[m_aa]])
    aa_dst = torch.cat([T_AA_O[m_aa], T_AA_I[m_aa]])
    e_co = torch.stack([aa_src, aa_dst], dim=0)
    if device is not None:
        e_co = e_co.to(device, non_blocking=True)
    data["wallet", "co_tx", "wallet"].edge_index = e_co

    # tx<->tx within-step: undirected
    m_tt = (T_TT_T <= k)
    tt_src = torch.cat([T_TT_A[m_tt], T_TT_B[m_tt]])
    tt_dst = torch.cat([T_TT_B[m_tt], T_TT_A[m_tt]])
    e_chain = torch.stack([tt_src, tt_dst], dim=0)
    if device is not None:
        e_chain = e_chain.to(device, non_blocking=True)
    data["tx", "tx_chain", "tx"].edge_index = e_chain
    return data


def edge_summary(data: HeteroData) -> str:
    parts = []
    for et in data.edge_types:
        n = data[et].edge_index.size(1)
        if n: parts.append(f"{et[0][:1]}-{et[1][:8]}-{et[2][:1]}: {n:,}")
    return " | ".join(parts)


for k in [1, 10, TRAIN_END, VAL_END, N_TIMESTEPS]:
    g = build_snapshot(k)
    print(f"snapshot at k={k:>2}: tx.x={tuple(g['tx'].x.shape)}  wallet.x={tuple(g['wallet'].x.shape)}  {edge_summary(g)}")

GRAPH_METADATA = build_snapshot(TRAIN_END).metadata()
print(f"metadata: node_types={GRAPH_METADATA[0]}  edge_types={len(GRAPH_METADATA[1])}")


In [ ]:
# Cell 7: Relational Graph Transformer (HGTConv-based)
# Same architecture as rgt.ipynb. Input dim is F_TX (raw 182) since the
# baseline does not use the engineered (traj+PPR) tx features.

class HGTBlock(nn.Module):
    def __init__(self, hidden: int, metadata, heads: int = HEADS,
                 dropout: float = DROPOUT, attn_dropout: float = ATTN_DROPOUT):
        super().__init__()
        self.ln1  = nn.LayerNorm(hidden)
        self.attn = HGTConv(hidden, hidden, metadata, heads=heads)
        self.ln2  = nn.LayerNorm(hidden)
        self.ffn  = nn.Sequential(
            nn.Linear(hidden, 2 * hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * hidden, hidden),
        )
        self.drop = nn.Dropout(dropout)
        self.attn_drop = nn.Dropout(attn_dropout)

    def forward(self, x_dict, edge_index_dict):
        # Pre-LN attention.
        # HGTConv -> HeteroDictLinear -> pyg_lib.segment_matmul is dtype-strict
        # and incompatible with autocast bf16. Run attention in fp32; rest of
        # the block stays in bf16 for speed.
        x_pre = {k: self.ln1(v) for k, v in x_dict.items()}
        with torch.amp.autocast(device_type="cuda", enabled=False):
            x_pre_fp32 = {k: v.float() for k, v in x_pre.items()}
            attn = self.attn(x_pre_fp32, edge_index_dict)
        x_dict = {k: x_dict[k] + self.attn_drop(attn[k]) if k in attn else x_dict[k]
                  for k in x_dict}
        # Pre-LN FFN
        x_pre = {k: self.ln2(v) for k, v in x_dict.items()}
        x_dict = {k: x_dict[k] + self.drop(self.ffn(x_pre[k])) for k in x_dict}
        return x_dict


class RelationalGraphTransformer(nn.Module):
    def __init__(self, metadata, hidden=HIDDEN, n_layers=N_LAYERS, heads=HEADS,
                 dropout=DROPOUT, attn_dropout=ATTN_DROPOUT,
                 f_tx=F_TX, f_w=F_W):
        super().__init__()
        self.tx_in = nn.Linear(f_tx, hidden)
        self.w_in  = nn.Linear(f_w,  hidden)
        self.in_norm_tx = nn.LayerNorm(hidden)
        self.in_norm_w  = nn.LayerNorm(hidden)
        self.blocks = nn.ModuleList([
            HGTBlock(hidden, metadata, heads, dropout, attn_dropout)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, x_dict, edge_index_dict):
        x_dict = {
            "tx":     self.in_norm_tx(self.tx_in(x_dict["tx"])),
            "wallet": self.in_norm_w(self.w_in(x_dict["wallet"])),
        }
        for blk in self.blocks:
            x_dict = blk(x_dict, edge_index_dict)
        return self.classifier(self.out_norm(x_dict["tx"])).squeeze(-1)


def _smoke():
    d = build_snapshot(TRAIN_END)
    m = RelationalGraphTransformer(d.metadata())
    out = m(d.x_dict, d.edge_index_dict)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  RGT[baseline]: out={tuple(out.shape)} params={n_params:,}")
_smoke()


In [ ]:
# Cell 7b: Leakage sanity check

def check_snapshot_strict(k: int):
    # Return (ok, errors).
    g = build_snapshot(k)
    errors = []
    edge_checks = [
        (("wallet", "sends_to", "tx"),       T_AI_T, lambda n: n),
        (("tx", "rev_sends_to", "wallet"),   T_AI_T, lambda n: n),
        (("tx", "pays_to", "wallet"),        T_AO_T, lambda n: n),
        (("wallet", "rev_pays_to", "tx"),    T_AO_T, lambda n: n),
        (("wallet", "co_tx", "wallet"),      T_AA_T, lambda n: 2 * n),
        (("tx", "tx_chain", "tx"),           T_TT_T, lambda n: 2 * n),
    ]
    for et, t_master, count_fn in edge_checks:
        n = g[et].edge_index.size(1)
        n_master = int((t_master <= k).sum())
        n_expected = count_fn(n_master)
        if n != n_expected:
            errors.append(f"edge_count {et}: snapshot has {n}, expected {n_expected}")
    wf_g = g["wallet"].x.numpy()
    wf_ref = wallet_features_at_T(k)
    if not np.allclose(wf_g, wf_ref, atol=1e-6):
        errors.append(f"wallet feature mismatch at k={k}")
    return (len(errors) == 0), errors


print("Leakage sanity check on snapshots at k in [1, 10, 20, 29, 34, 49]:")
for k in [1, 10, 20, 29, 34, 49]:
    ok, errs = check_snapshot_strict(k)
    if ok:
        print(f"  k={k:>2}  OK  (all edges <= {k}, wallet features = wallet_features_at_T({k}))")
    else:
        print(f"  k={k:>2}  FAIL")
        for e in errs[:3]:
            print(f"     - {e}")


In [ ]:
# Cell 8: STRICT per-target causal training utility (NeighborLoader-sampled, bf16)
# Identical to rgt.ipynb cell 8.

def _bce(logits, y, pos_weight_t):
    return F.binary_cross_entropy_with_logits(
        logits.float(), y.float(), pos_weight=pos_weight_t,
    )


def _calibrate_threshold(y_true, p_score):
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return 0.5
    prec, rec, thr = precision_recall_curve(y_true, p_score)
    f1 = (2 * prec * rec / (prec + rec + 1e-12))[:-1]
    return float(thr[int(np.argmax(f1))]) if len(f1) else 0.5


def _pos_weight_from(mask: np.ndarray) -> float:
    y = tx_label[mask]
    pi = float((y == 1).mean())
    pi = max(min(pi, 0.5), 1e-3)
    return (1 - pi) / pi


def _attach_y(g_cpu):
    g_cpu["tx"].y = torch.from_numpy(tx_label).long()
    return g_cpu


def _make_loader(g_cpu, seed_mask: np.ndarray, batch_size: int, fanout, shuffle: bool):
    g_cpu = _attach_y(g_cpu)
    return NeighborLoader(
        g_cpu, num_neighbors=fanout, batch_size=batch_size,
        input_nodes=("tx", torch.from_numpy(seed_mask)), shuffle=shuffle,
    )


def _eval_per_timestep_sampled(model, eval_label_mask, fanout, batch_size):
    model.eval()
    p_pool, y_pool = [], []
    with torch.no_grad():
        for k in range(1, N_TIMESTEPS + 1):
            mask_k = eval_label_mask & (tx_t_arr == k)
            if not mask_k.any():
                continue
            g_k = build_snapshot(k)
            loader = _make_loader(g_k, mask_k, batch_size, fanout, shuffle=False)
            for batch in loader:
                batch = batch.to(DEVICE, non_blocking=True)
                with amp_ctx():
                    logits = model(batch.x_dict, batch.edge_index_dict)
                n_seed = batch["tx"].batch_size
                p_pool.append(torch.sigmoid(logits[:n_seed].float()).cpu().numpy())
                y_pool.append(batch["tx"].y[:n_seed].cpu().numpy())
            del g_k, loader
    free_cuda()
    if p_pool:
        return np.concatenate(p_pool), np.concatenate(y_pool)
    return np.array([]), np.array([])


def train_inductive(
    train_T_cutoff: int,
    train_label_mask: np.ndarray,
    val_label_mask: np.ndarray,
    test_label_mask: np.ndarray,
    n_epochs: int,
    seed: int = RANDOM_SEED,
    lr: float = LR,
    patience: int = PATIENCE,
    fanout=None,
    batch_size: int = None,
    verbose: bool = False,
):
    if fanout is None:     fanout = SAMPLE_FANOUT
    if batch_size is None: batch_size = SAMPLE_BATCH

    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    pos_weight = _pos_weight_from(train_label_mask)
    pos_weight_t = torch.tensor(pos_weight, dtype=torch.float, device=DEVICE)

    train_ts = sorted({int(k) for k in tx_t_arr[train_label_mask]
                       if 1 <= int(k) <= train_T_cutoff})
    if not train_ts:
        raise ValueError("No labelled training timesteps within [1, train_T_cutoff]")

    g_init = build_snapshot(train_T_cutoff)
    metadata = g_init.metadata()
    init_loader = _make_loader(g_init, train_label_mask, batch_size, fanout, shuffle=False)
    init_batch  = next(iter(init_loader)).to(DEVICE, non_blocking=True)

    model = RelationalGraphTransformer(metadata).to(DEVICE)
    with torch.no_grad():
        _ = model(init_batch.x_dict, init_batch.edge_index_dict)
    del init_batch, init_loader, g_init

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    best_val_f1 = -1.0
    best_state = None
    best_thr = 0.5
    bad = 0

    for epoch in range(n_epochs):
        model.train()
        rng = np.random.default_rng(seed + epoch)
        ts_order = list(train_ts); rng.shuffle(ts_order)
        epoch_loss, n_batches = 0.0, 0
        for k in ts_order:
            seeds_at_k = train_label_mask & (tx_t_arr == k)
            if not seeds_at_k.any(): continue
            g_k = build_snapshot(k)
            loader = _make_loader(g_k, seeds_at_k, batch_size, fanout, shuffle=True)
            for batch in loader:
                batch = batch.to(DEVICE, non_blocking=True)
                opt.zero_grad()
                with amp_ctx():
                    logits = model(batch.x_dict, batch.edge_index_dict)
                    n_seed = batch["tx"].batch_size
                    loss = _bce(logits[:n_seed], batch["tx"].y[:n_seed], pos_weight_t)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
                epoch_loss += float(loss.detach())
                n_batches += 1
            del g_k, loader
        avg_loss = epoch_loss / max(n_batches, 1)

        val_p, val_y = _eval_per_timestep_sampled(model, val_label_mask, fanout, batch_size)
        if val_p.size > 0:
            thr = _calibrate_threshold(val_y, val_p)
            f1_val = f1_score(val_y, (val_p >= thr).astype(np.int64), pos_label=1, zero_division=0)
        else:
            thr, f1_val = 0.5, 0.0

        improved = f1_val > best_val_f1 + 1e-4
        if improved:
            best_val_f1 = f1_val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_thr = thr
            bad = 0
        else:
            bad += 1
        if verbose:
            print(f"   ep{epoch+1:>3} loss={avg_loss:.4f} val_f1={f1_val:.4f} thr={thr:.3f}{' *' if improved else ''}")
        if bad >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_p, test_y = _eval_per_timestep_sampled(model, test_label_mask, fanout, batch_size)
    if test_p.size > 0:
        f1    = f1_score(test_y, (test_p >= best_thr).astype(np.int64), pos_label=1, zero_division=0)
        f1_05 = f1_score(test_y, (test_p >= 0.5).astype(np.int64),     pos_label=1, zero_division=0)
        auc   = roc_auc_score(test_y, test_p) if test_y.sum() > 0 and test_y.sum() < len(test_y) else float("nan")
        prauc = average_precision_score(test_y, test_p) if test_y.sum() > 0 else float("nan")
    else:
        f1 = f1_05 = auc = prauc = float("nan")

    del model, opt
    free_cuda()

    return {
        "f1": f1, "f1_05": f1_05, "auc": auc, "prauc": prauc,
        "thr": best_thr, "best_val_f1": best_val_f1,
        "n_train": int(train_label_mask.sum()),
        "n_val":   int(val_label_mask.sum()),
        "n_test":  int(test_label_mask.sum()),
        "p_test": test_p, "y_test": test_y,
        "state": best_state,
        "pos_weight": pos_weight,
    }


## Strict-inductive evaluation

In [ ]:
# Cell 10: run strict-inductive

labelled = (tx_label != -1)
m_train = labelled & (tx_t_arr <= TRAIN_END)
m_val   = labelled & (tx_t_arr >= TRAIN_END + 1) & (tx_t_arr <= VAL_END)
m_test  = labelled & (tx_t_arr >= TEST_START)
print(f"labels: train(t<=29)={m_train.sum():,}  val[30..34]={m_val.sum():,}  test[35..49]={m_test.sum():,}")

t0 = time.time()
strict_res = train_inductive(
    train_T_cutoff=TRAIN_END,
    train_label_mask=m_train, val_label_mask=m_val, test_label_mask=m_test,
    n_epochs=N_EPOCHS_STRICT, seed=RANDOM_SEED, patience=PATIENCE, verbose=True,
)
print(f"\n[strict] RGT[baseline]  F1={strict_res['f1']:.4f}  F1@0.5={strict_res['f1_05']:.4f}  "
      f"AUC={strict_res['auc']:.4f}  PR-AUC={strict_res['prauc']:.4f}  thr={strict_res['thr']:.3f}  "
      f"({time.time()-t0:.0f}s)")


## Walk-forward evaluation

In [ ]:
# Cell 11: run walk-forward

walk_per_T = {}
walk_pooled_p = []
walk_pooled_y = []

t_overall = time.time()
for T in TEST_TIMESTEPS:
    v_lo = max(1, T - WALK_VAL_WIN)
    m_val_T   = labelled & (tx_t_arr >= v_lo) & (tx_t_arr <= T - 1)
    m_train_T = labelled & (tx_t_arr <= T - 1) & ~m_val_T
    m_test_T  = labelled & (tx_t_arr == T)
    if m_test_T.sum() == 0 or m_train_T.sum() == 0:
        continue
    tic = time.time()
    res = train_inductive(
        train_T_cutoff=T - 1,
        train_label_mask=m_train_T, val_label_mask=m_val_T, test_label_mask=m_test_T,
        n_epochs=N_EPOCHS_WALK, seed=RANDOM_SEED + T, patience=PATIENCE,
    )
    walk_per_T[T] = {"f1": res["f1"], "f1_05": res["f1_05"], "auc": res["auc"],
                     "prauc": res["prauc"], "n_test": res["n_test"], "thr": res["thr"]}
    walk_pooled_p.append(res["p_test"]); walk_pooled_y.append(res["y_test"])
    print(f"  t*={T:>2}  F1={res['f1']:.4f}  F1@0.5={res['f1_05']:.4f}  "
          f"AUC={res['auc']:.4f}  PR-AUC={res['prauc']:.4f}  ({time.time()-tic:.0f}s)")
    free_cuda()

if walk_pooled_p:
    P = np.concatenate(walk_pooled_p); Y = np.concatenate(walk_pooled_y)
    thr = _calibrate_threshold(Y, P)
    walk_pooled = {
        "f1":    f1_score(Y, (P >= thr).astype(np.int64),  pos_label=1, zero_division=0),
        "f1_05": f1_score(Y, (P >= 0.5).astype(np.int64),  pos_label=1, zero_division=0),
        "auc":   roc_auc_score(Y, P),
        "prauc": average_precision_score(Y, P),
        "thr":   thr, "n_test": int(len(Y)),
    }
else:
    walk_pooled = {"f1": float("nan"), "f1_05": float("nan"), "auc": float("nan"),
                   "prauc": float("nan"), "thr": 0.5, "n_test": 0}

print(f"\n[walk-forward pooled]  F1={walk_pooled['f1']:.4f}  F1@0.5={walk_pooled['f1_05']:.4f}  "
      f"AUC={walk_pooled['auc']:.4f}  PR-AUC={walk_pooled['prauc']:.4f}  "
      f"({time.time()-t_overall:.0f}s total)")


In [ ]:
# Cell 12: final summary

print("=" * 90)
print("RELATIONAL GRAPH TRANSFORMER -- BASELINE  (raw tx + causal wallet features only)")
print("=" * 90)
print(f"{'protocol':<25}  {'F1[cal]':>8}  {'F1@0.5':>8}  {'AUC':>7}  {'PR-AUC':>7}  {'thr':>6}")
print("-" * 90)
print(f"{'strict-inductive':<25}  "
      f"{strict_res['f1']:>8.4f}  {strict_res['f1_05']:>8.4f}  "
      f"{strict_res['auc']:>7.4f}  {strict_res['prauc']:>7.4f}  {strict_res['thr']:>6.3f}")
print(f"{'walk-forward pooled':<25}  "
      f"{walk_pooled['f1']:>8.4f}  {walk_pooled['f1_05']:>8.4f}  "
      f"{walk_pooled['auc']:>7.4f}  {walk_pooled['prauc']:>7.4f}  {walk_pooled['thr']:>6.3f}")

print()
print("Walk-forward per-timestep F1:")
print(f"  {'t*':>3} {'n_test':>6} {'F1[cal]':>8} {'F1@0.5':>8} {'AUC':>7} {'PR-AUC':>7}")
for T in TEST_TIMESTEPS:
    if T not in walk_per_T: continue
    r = walk_per_T[T]
    print(f"  {T:>3} {r['n_test']:>6} {r['f1']:>8.4f} {r['f1_05']:>8.4f} {r['auc']:>7.4f} {r['prauc']:>7.4f}")
